In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

df = pd.read_csv("AB_NYC_2019.csv")

In [ ]:
display(df.head(10))
display(df.sample(5, random_state=42))

print("Rows and columns:", df.shape)
print("Total cells:", df.size)

df.info()
display(df.dtypes)

print("Numerical columns:")
print(df.select_dtypes(include="number").columns.tolist())

print("Categorical/text columns:")
print(df.select_dtypes(include=["object", "category"]).columns.tolist())

In [ ]:
print("Unique IDs:", df["id"].nunique())
print("Total rows:", len(df))
print("Duplicate IDs:", df["id"].duplicated().sum())
print("All IDs unique:", df["id"].is_unique)

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate listing IDs:", df["id"].duplicated().sum())

duplicate_ids = df[df.duplicated("id", keep=False)].sort_values("id")
display(duplicate_ids.head(20))

In [ ]:
display(
    df.groupby("number_of_reviews", dropna=False)
      .agg(
          listing_count=("id", "size"),
          missing_last_review=("last_review", lambda s: s.isna().sum()),
          missing_reviews_per_month=(
              "reviews_per_month", lambda s: s.isna().sum()
          )
      )
      .sort_values("listing_count", ascending=False)
      .head(10)
)

print("Listings with zero reviews:",
      (df["number_of_reviews"] == 0).sum())

print("Zero-review listings with missing last_review:",
      ((df["number_of_reviews"] == 0) & df["last_review"].isna()).sum())

print("Zero-review listings with missing reviews_per_month:",
      ((df["number_of_reviews"] == 0) & df["reviews_per_month"].isna()).sum())

In [ ]:
display(df[["price", "minimum_nights"]].describe())

print("Zero-priced listings:", (df["price"] == 0).sum())

price_q1 = df["price"].quantile(0.25)
price_q3 = df["price"].quantile(0.75)
price_iqr = price_q3 - price_q1

upper_price_fence = price_q3 + 1.5 * price_iqr

potential_price_outliers = df[df["price"] > upper_price_fence]

print("Potential high-price outliers:", len(potential_price_outliers))
display(
    potential_price_outliers[
        ["name", "neighbourhood_group", "room_type", "price"]
    ].sort_values("price", ascending=False).head(10)
)

In [ ]:
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

print(df["last_review"].dtype)

In [ ]:
cleaned = df.copy()

cleaned = cleaned.drop_duplicates(subset="id", keep="first")

cleaned = cleaned.loc[cleaned["price"] > 0].copy()

print("Original rows:", len(df))
print("Rows after cleaning:", len(cleaned))
print("Rows removed:", len(df) - len(cleaned))

In [ ]:
cleaned["price_category"] = pd.cut(
    cleaned["price"],
    bins=[0, 100, 250, np.inf],
    labels=["Low", "Medium", "High"],
    right=True,
    include_lowest=True
)

cleaned["availability_category"] = pd.cut(
    cleaned["availability_365"],
    bins=[-1, 0, 180, 365],
    labels=["Unavailable", "Low availability", "High availability"]
)

cleaned["estimated_cost"] = (
    cleaned["price"] * cleaned["minimum_nights"]
)

display(
    cleaned[
        [
            "price", "price_category",
            "availability_365", "availability_category",
            "minimum_nights", "estimated_cost"
        ]
    ].head(10)
)

In [ ]:
overall_median_price = cleaned["price"].median()

manhattan_affordable = cleaned.loc[
    (cleaned["neighbourhood_group"] == "Manhattan")
    & (cleaned["room_type"] == "Entire home/apt")
    & (cleaned["price"] < overall_median_price),
    ["name", "neighbourhood", "room_type", "price"]
]

print("Overall median price:", overall_median_price)
print("Matching listings:", len(manhattan_affordable))
display(manhattan_affordable.head(10))

In [ ]:
long_stay_popular = cleaned.loc[
    (cleaned["minimum_nights"] >= 7)
    & (cleaned["number_of_reviews"] > 50),
    ["name", "neighbourhood_group", "price",
     "minimum_nights", "number_of_reviews"]
]

print("Matching listings:", len(long_stay_popular))
display(long_stay_popular.head(10))

In [ ]:
top_10_expensive = (
    cleaned.sort_values("price", ascending=False)
    [["name", "neighbourhood_group", "room_type",
      "price", "minimum_nights"]]
    .head(10)
)

display(top_10_expensive)

In [ ]:
host_listing_counts = (
    cleaned.groupby("host_id")
    .size()
    .sort_values(ascending=False)
)

hosts_over_5 = host_listing_counts[host_listing_counts > 5]

print("Hosts with more than 5 listings:", len(hosts_over_5))
display(hosts_over_5.head(10))

In [ ]:
review_comparison = cleaned.loc[
    cleaned["number_of_reviews"].eq(0)
    | cleaned["number_of_reviews"].ge(100)
].copy()

review_comparison["review_group"] = np.where(
    review_comparison["number_of_reviews"].eq(0),
    "Zero reviews",
    "100+ reviews"
)

display(
    review_comparison.groupby("review_group")
    .agg(
        listing_count=("id", "size"),
        mean_availability=("availability_365", "mean"),
        median_availability=("availability_365", "median"),
        mean_price=("price", "mean"),
        median_price=("price", "median")
    )
)

In [ ]:
borough_summary = (
    cleaned.groupby("neighbourhood_group")
    .agg(
        listing_count=("id", "size"),
        mean_price=("price", "mean"),
        median_price=("price", "median"),
        median_minimum_nights=("minimum_nights", "median")
    )
    .sort_values("listing_count", ascending=False)
    .round(2)
)

display(borough_summary)

In [ ]:
borough_room_summary = (
    cleaned.groupby(["neighbourhood_group", "room_type"])
    .agg(
        listing_count=("id", "size"),
        median_price=("price", "median"),
        mean_price=("price", "mean")
    )
    .sort_values(
        ["neighbourhood_group", "median_price"],
        ascending=[True, False]
    )
    .round(2)
)

display(borough_room_summary)

In [ ]:
neighbourhood_summary = (
    cleaned.groupby(["neighbourhood_group", "neighbourhood"])
    .agg(
        listing_count=("id", "size"),
        median_price=("price", "median")
    )
    .reset_index()
)

eligible_neighbourhoods = neighbourhood_summary.loc[
    neighbourhood_summary["listing_count"] >= 30
]

top_10_neighbourhoods = (
    eligible_neighbourhoods
    .sort_values("median_price", ascending=False)
    .head(10)
    .round(2)
)

display(top_10_neighbourhoods)

In [ ]:
top_10_hosts = (
    cleaned.groupby("host_id")
    .agg(
        listing_count=("id", "size"),
        median_price=("price", "median")
    )
    .sort_values("listing_count", ascending=False)
    .head(10)
)

display(top_10_hosts)

In [ ]:
room_price_summary = (
    cleaned.groupby("room_type")
    .agg(
        listing_count=("id", "size"),
        mean_price=("price", "mean"),
        median_price=("price", "median")
    )
    .sort_values("median_price", ascending=False)
    .round(2)
)

display(room_price_summary)

In [ ]:
median_price_pivot = pd.pivot_table(
    cleaned,
    index="neighbourhood_group",
    columns="room_type",
    values="price",
    aggfunc="median",
    observed=False
)

display(median_price_pivot.round(2))

In [ ]:
listing_count_pivot = pd.pivot_table(
    cleaned,
    index="neighbourhood_group",
    columns="room_type",
    values="id",
    aggfunc="count",
    fill_value=0,
    observed=False
)

display(listing_count_pivot)

In [ ]:
highest_median_room = median_price_pivot.idxmax(axis=1)
highest_median_value = median_price_pivot.max(axis=1)

highest_room_summary = pd.DataFrame({
    "room_type_with_highest_median": highest_median_room,
    "median_price": highest_median_value
})

display(highest_room_summary.round(2))

In [ ]:
host_summary = (
    cleaned.groupby("host_id")
    .agg(
        host_listing_count=("id", "size"),
        host_median_price=("price", "median")
    )
    .reset_index()
)

display(host_summary.head())

In [ ]:
rows_before = len(cleaned)

merged = cleaned.merge(
    host_summary,
    on="host_id",
    how="left",
    validate="many_to_one"
)

rows_after = len(merged)

print("Rows before merge:", rows_before)
print("Rows after merge:", rows_after)
print("Row count preserved:", rows_before == rows_after)

display(
    merged[
        ["id", "host_id", "price",
         "host_listing_count", "host_median_price"]
    ].head(10)
)

In [ ]:
merged["host_type"] = np.where(
    merged["host_listing_count"] == 1,
    "Single-listing host",
    "Multi-listing host"
)

host_type_summary = (
    merged.groupby("host_type")
    .agg(
        listing_count=("id", "size"),
        median_listing_price=("price", "median"),
        mean_listing_price=("price", "mean"),
        median_availability=("availability_365", "median"),
        median_reviews=("number_of_reviews", "median")
    )
    .round(2)
)

display(host_type_summary)

In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
borough_counts = (
    cleaned["neighbourhood_group"]
    .value_counts()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
sns.barplot(x=borough_counts.index, y=borough_counts.values)

plt.title("Number of Airbnb Listings by Borough")
plt.xlabel("Borough")
plt.ylabel("Number of listings")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plot_data = cleaned.loc[cleaned["price"] <= cleaned["price"].quantile(0.95)]

plt.figure(figsize=(10, 6))
sns.boxplot(data=plot_data, x="room_type", y="price")

plt.title("Airbnb Price Distribution by Room Type (Up to 95th Percentile)")
plt.xlabel("Room type")
plt.ylabel("Nightly price (USD)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
median_by_borough = (
    cleaned.groupby("neighbourhood_group")["price"]
    .median()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
sns.barplot(x=median_by_borough.index, y=median_by_borough.values)

plt.title("Median Airbnb Price by Borough")
plt.xlabel("Borough")
plt.ylabel("Median nightly price (USD)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
scatter_data = cleaned.sample(
    n=min(5000, len(cleaned)),
    random_state=42
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=scatter_data,
    x="number_of_reviews",
    y="price",
    hue="room_type",
    alpha=0.6
)

plt.title("Nightly Price versus Number of Reviews")
plt.xlabel("Number of reviews")
plt.ylabel("Nightly price (USD)")
plt.tight_layout()
plt.show()